In [66]:
import chess
import chess.pgn
import torch
import torch.nn as nn
import torch.nn.functional as f
import pygame

if torch.cuda.is_available():
	print("PyTorch is using the GPU")
	GPUCount = torch.cuda.device_count()
	print(f"Found {GPUCount} GPUs")

	for i in range(GPUCount):
		print(f"GPU {i} found: {torch.cuda.get_device_name(i)}")

	device = torch.device("cuda:0")
else:
	print("PyTorch is using the CPU")
	device = torch.device("cpu")

print(f"Selected Device: {device}")

PyTorch is using the GPU
Found 1 GPUs
GPU 0 found: NVIDIA GeForce RTX 5070 Laptop GPU
Selected Device: cuda:0


In [67]:
pygame.init()
BoardSize = 512
SquareSize = BoardSize//8
DarkColor = (90,90,90)
LightColor = (205,205,205)
ChessBoard = chess.Board()
ChessBoardGUI = pygame.display.set_mode((BoardSize,BoardSize))
turn = 0
images = {}
ranks = [8,7,6,5,4,3,2,1]
files = ['a','b','c','d','e','f','g','h']
EvalCache = {}
MVVLVALookup = [[0,0,0,0,0,0,0],
				[0,9,29,29,49,89,0],
				[0,7,27,27,47,87,0],
				[0,7,27,27,47,87,0],
				[0,5,25,25,45,85,0],
				[0,1,21,21,41,81,0],
				[0,1,21,21,41,81,0]
				]

In [68]:
def LoadImages(SquareSize):
	pieces = {'K','Q','B','N','R','P','k','q','b','n','r','p'}
	ImgMap = {'K':'WhiteKing','Q':'WhiteQueen','B':'WhiteBishop','N':'WhiteKnight','R':'WhiteRook','P':'WhitePawn','k':'BlackKing','q':'BlackQueen','b':'BlackBishop','n':'BlackKnight','r':'BlackRook','p':'BlackPawn'}

	for piece in pieces:
		img = pygame.image.load(f'ChessSprites/{ImgMap[piece]}.png')
		img = pygame.transform.scale(img,(SquareSize,SquareSize))
		images[piece] = img
	return

In [69]:
def DrawBoard(ChessBoardGUI,SquareSize):
	for row in range(8):
		for col in range(8):
			x = col*SquareSize
			y = row*SquareSize
			Color = LightColor if ((row+col)%2==0) else DarkColor
			pygame.draw.rect(ChessBoardGUI,Color,(x,y,SquareSize,SquareSize))
	return

In [70]:
def DrawPieces(ChessBoardGUI,ChessBoard):
	for row in range(8):
		for col in range(8):
			x = col*SquareSize
			y = row*SquareSize
			SquareIdx = chess.square(col,7-row)
			piece = ChessBoard.piece_at(SquareIdx)
			if (piece is not None):
				ChessBoardGUI.blit(images[piece.symbol()],(x,y))

In [71]:
def ProcessChessData(FENString):
	tensor = torch.zeros((16,8,8), dtype=torch.float32)
	board = chess.Board(FENString)

	PieceToLayer = {
		'P':0,'N':1,'B':2,'R':3,'Q':4,'K':5,
		'p':6,'n':7,'b':8,'r':9,'q':10,'k':11
	}

	for square in chess.SQUARES:
		piece = board.piece_at(square)

		if piece:
			symbol = piece.symbol()
			layer = PieceToLayer[symbol]

			row = 7-(square//8)
			col = square%8

			tensor[layer,row,col] = 1.0

	if board.turn == chess.WHITE:
		tensor[12,:,:] = 1.0

	if board.has_kingside_castling_rights(chess.WHITE):
		tensor[13,7,7] = 1.0
	if board.has_queenside_castling_rights(chess.WHITE):
		tensor[13,7,0] = 1.0
	if board.has_kingside_castling_rights(chess.BLACK):
		tensor[13,0,7] = 1.0
	if board.has_queenside_castling_rights(chess.BLACK):
		tensor[13,0,0] = 1.0

	for i in range(8):
		tensor[14,i,:] = (1.0/7)*(i)
		tensor[15,:,i] = (1.0/7)*(i)

	return tensor

In [72]:
class SEBlock(nn.Module):
	def __init__(self,channels,reduction=16):
		super().__init__()
		self.squeeze = nn.AdaptiveAvgPool2d(1)
		self.excite = nn.Sequential(
			nn.Linear(channels,channels//reduction,bias=False),
			nn.SiLU(inplace=True),
			nn.Linear(channels//reduction,channels,bias=False),
			nn.Sigmoid()
		)

	def forward(self,x):
		b,c,_,_ = x.size()
		y = self.squeeze(x).view(b,c)
		y = self.excite(y).view(b,c,1,1)
		return x * y.expand_as(x)

class ResidualBlock(nn.Module):
	def __init__(self,NumChannels):
		super().__init__()
		self.conv1 = nn.Conv2d(NumChannels,NumChannels,kernel_size=3,padding=1)
		self.bn1 = nn.BatchNorm2d(NumChannels)

		self.conv2 = nn.Conv2d(NumChannels,NumChannels,kernel_size=3,padding=1)
		self.bn2 = nn.BatchNorm2d(NumChannels)

		self.se = SEBlock(NumChannels)

	def forward(self,x):
		residual = x
		
		x = f.silu(self.bn1(self.conv1(x)))

		x = self.bn2(self.conv2(x))

		x = self.se(x)

		x += residual

		return f.silu(x)
	
class ChessNet(nn.Module):
	def __init__(self):
		super().__init__()

		self.ConvInput = nn.Conv2d(in_channels=16,out_channels=256,kernel_size=3,padding=1)
		self.BnInput = nn.BatchNorm2d(256)

		self.ResTower = nn.Sequential(*[ResidualBlock(256) for _ in range(10)])

		self.ConvValue = nn.Conv2d(in_channels=256,out_channels=32,kernel_size=1)
		self.BnValue = nn.BatchNorm2d(32)

		self.flat = nn.Flatten()

		self.fc1 = nn.Linear(32*8*8,256)
		self.fc2 = nn.Linear(256,1)

	def forward(self,x):
		x = f.silu(self.BnInput(self.ConvInput(x)))

		x = self.ResTower(x)

		x = f.silu(self.BnValue(self.ConvValue(x)))
		x = self.flat(x)
		x = f.silu(self.fc1(x))

		x = torch.tanh(self.fc2(x))

		return x
	
WhiteModel = ChessNet()
BlackModel = ChessNet()
WhiteModel.to(device)
BlackModel.to(device)
WhiteModel.load_state_dict(torch.load('ChessModel.pth'))
BlackModel.load_state_dict(torch.load('ChessModel.pth'))
WhiteModel.eval()
BlackModel.eval()

ChessNet(
  (ConvInput): Conv2d(16, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (BnInput): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (ResTower): Sequential(
    (0): ResidualBlock(
      (conv1): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (bn1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (bn2): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (se): SEBlock(
        (squeeze): AdaptiveAvgPool2d(output_size=1)
        (excite): Sequential(
          (0): Linear(in_features=256, out_features=16, bias=False)
          (1): SiLU(inplace=True)
          (2): Linear(in_features=16, out_features=256, bias=False)
          (3): Sigmoid()
        )
      )
    )
    (1): ResidualBlock(
      (conv1): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1

In [73]:
def MoveSort(move):
	if ChessBoard.is_en_passant(move):
		return 9
	elif ChessBoard.is_capture(move):
		return MVVLVALookup[ChessBoard.piece_at(move.from_square).piece_type][ChessBoard.piece_at(move.to_square).piece_type]
	else:
		return 0

In [74]:
def EvalNetwork(board,model,depth):
	fen = board.fen().split()
	fen = ' '.join(fen[:4])
	if fen in EvalCache:
		return EvalCache[fen]
	
	if board.is_checkmate():
		return -(9999.0+depth) if board.turn == chess.WHITE else (9999.0+depth)
	if board.is_game_over():
		return 0.0
	
	tensor = ProcessChessData(fen).unsqueeze(0).to(device)

	with torch.no_grad():
		score = model(tensor).item()

	EvalCache[fen] = score

	return score

In [75]:
def QuiscenceSearch(board,alpha,beta,MaximisingPlayer,model):

	StandPat = EvalNetwork(board,model,0)

	if MaximisingPlayer:
		if StandPat > beta:
			return beta
	
		elif StandPat > alpha:
			alpha = StandPat
	else:
		if StandPat < alpha:
			return alpha
		
		elif StandPat < beta:
			beta = StandPat

	ChaoticMoves = []

	for move in board.legal_moves:
		if board.is_en_passant(move) or board.is_capture(move) or move.promotion is not None:
			ChaoticMoves.append(move)

	ChaoticMoves = sorted(ChaoticMoves,key=MoveSort,reverse=True)

	if MaximisingPlayer:
		MaxEval = StandPat
		for move in ChaoticMoves:
			board.push(move)
			EvalScore = QuiscenceSearch(board,alpha,beta,False,model)
			board.pop()

			MaxEval = max(MaxEval,EvalScore)
			alpha = max(alpha,MaxEval)

			if beta<=alpha:
				break
		return MaxEval
	else:
		MinEval = StandPat
		for move in ChaoticMoves:
			board.push(move)
			EvalScore = QuiscenceSearch(board,alpha,beta,True,model)
			board.pop()

			MinEval = min(MinEval,EvalScore)
			beta = min(beta,EvalScore)

			if beta <= alpha:
				break

		return MinEval

In [76]:
def minimax(board,depth,alpha,beta,MaximisingPlayer,model):
	if board.is_game_over() or board.can_claim_draw():
		return EvalNetwork(board,model,depth)
	
	if depth == 0:
		return QuiscenceSearch(board,alpha,beta,MaximisingPlayer,model)
	
	if MaximisingPlayer:
		MaxEval = -float('inf')
		for move in board.legal_moves:
			board.push(move)
			EvalScore = minimax(board,depth-1,alpha,beta,False,model)
			board.pop()

			MaxEval = max(MaxEval,EvalScore)
			alpha = max(alpha,EvalScore)

			if beta <= alpha:
				break

		return MaxEval
	
	else:
		MinEval = float('inf')
		for move in board.legal_moves:
			board.push(move)
			EvalScore = minimax(board,depth-1,alpha,beta,True,model)
			board.pop()

			MinEval = min(MinEval,EvalScore)
			beta = min(beta,EvalScore)

			if beta <= alpha:
				break

		return MinEval

In [77]:
def GetBestMove(board,depth,model):
	BestMove = None
	MaximisingPlayer = (board.turn == chess.WHITE)

	BestEval = -float('inf') if MaximisingPlayer else float('inf')
	alpha = -float('inf')
	beta = float('inf')

	MoveList = sorted(board.legal_moves,key=MoveSort,reverse=True)

	for move in MoveList:
		board.push(move)
		MoveEval = minimax(board,depth-1,alpha,beta,not MaximisingPlayer,model)
		board.pop()

		if MaximisingPlayer:
			if MoveEval > BestEval:
				BestEval = MoveEval
				BestMove = move
			alpha = max(alpha,MoveEval)
		else:
			if MoveEval < BestEval:
				BestEval = MoveEval
				BestMove = move
			beta = min(beta,MoveEval)

	return BestMove

In [78]:
LoadImages(SquareSize)
PlayerClicks = []

try:
	with torch.no_grad():
		while not ChessBoard.is_game_over():
			if turn == 0:
				BestMove = GetBestMove(ChessBoard,3,WhiteModel)
				if BestMove is not None:
					ChessBoard.push(BestMove)

				turn = 1
		
			DrawBoard(ChessBoardGUI,SquareSize)
			DrawPieces(ChessBoardGUI,ChessBoard)
			pygame.display.flip()
			
			if turn == 1:
				BestMove = GetBestMove(ChessBoard,3,BlackModel)
				if BestMove is not None:
					ChessBoard.push(BestMove)

				turn = 0

			DrawBoard(ChessBoardGUI,SquareSize)
			DrawPieces(ChessBoardGUI,ChessBoard)
			pygame.display.flip()
		
		result = ChessBoard.outcome()
		
		if result.winner is True:
			print("White wins!")
		elif result.winner is False:
			print("Black wins!")
		else:
			print("Draw!")
			
		print(f"Reason: {result.termination.name}")

		GameRecord = chess.pgn.Game.from_board(ChessBoard)

		with open('BotvBotV2.pgn','w') as RecordFile:
			print(GameRecord,file=RecordFile,end='\n\n')
		print('Recorded!')
finally:
	pygame.quit()

KeyboardInterrupt: 